In [1]:
import pandas as pd
import numpy as np
from db import get_db_engine

In [2]:
df = pd.read_csv('customer_shopping_behavior.csv')

In [3]:
df.columns = df.columns.str.lower().str.replace(' ', '_')
df = df.rename(columns={'purchase_amount_(usd)': 'purchase_amount'})

In [4]:
df.head(3)

,customer_id,age,gender,item_purchased,category,purchase_amount,location,size,color,season,review_rating,subscription_status,shipping_type,discount_applied,promo_code_used,previous_purchases,payment_method,frequency_of_purchases
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Express,Yes,Yes,14,Venmo,Fortnightly
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Express,Yes,Yes,2,Cash,Fortnightly
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,Yes,Free Shipping,Yes,Yes,23,Credit Card,Weekly


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3900 entries, 0 to 3899
Data columns (total 18 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   customer_id             3900 non-null   int64  
 1   age                     3900 non-null   int64  
 2   gender                  3900 non-null   str    
 3   item_purchased          3900 non-null   str    
 4   category                3900 non-null   str    
 5   purchase_amount         3900 non-null   int64  
 6   location                3900 non-null   str    
 7   size                    3900 non-null   str    
 8   color                   3900 non-null   str    
 9   season                  3900 non-null   str    
 10  review_rating           3863 non-null   float64
 11  subscription_status     3900 non-null   str    
 12  shipping_type           3900 non-null   str    
 13  discount_applied        3900 non-null   str    
 14  promo_code_used         3900 non-null   str    
 15

In [6]:
# Summary statistics using .describe()
df.describe(include='all')

,customer_id,age,gender,item_purchased,category,purchase_amount,location,size,color,season,review_rating,subscription_status,shipping_type,discount_applied,promo_code_used,previous_purchases,payment_method,frequency_of_purchases
count,3900.000000,3900.000000,3900,3900,3900,3900.000000,3900,3900,3900,3900,3863.000000,3900,3900,3900,3900,3900.000000,3900,3900
unique,NaN,NaN,2,25,4,NaN,50,4,25,4,NaN,2,6,2,2,NaN,6,7
top,NaN,NaN,Male,Blouse,Clothing,NaN,Montana,M,Olive,Spring,NaN,No,Free Shipping,No,No,NaN,PayPal,Every 3 Months
freq,NaN,NaN,2652,171,1737,NaN,96,1755,177,999,NaN,2847,675,2223,2223,NaN,677,584
mean,1950.500000,44.068462,NaN,NaN,NaN,59.764359,NaN,NaN,NaN,NaN,3.750065,NaN,NaN,NaN,NaN,25.351538,NaN,NaN
std,1125.977353,15.207589,NaN,NaN,NaN,23.685392,NaN,NaN,NaN,NaN,0.716983,NaN,NaN,NaN,NaN,14.447125,NaN,NaN
min,1.000000,18.000000,NaN,NaN,NaN,20.000000,NaN,NaN,NaN,NaN,2.500000,NaN,NaN,NaN,NaN,1.000000,NaN,NaN
25%,975.750000,31.000000,NaN,NaN,NaN,39.000000,NaN,NaN,NaN,NaN,3.100000,NaN,NaN,NaN,NaN,13.000000,NaN,NaN
50%,1950.500000,44.000000,NaN,NaN,NaN,60.000000,NaN,NaN,NaN,NaN,3.800000,NaN,NaN,NaN,NaN,25.000000,NaN,NaN
75%,2925.250000,57.000000,NaN,NaN,NaN,81.000000,NaN,NaN,NaN,NaN,4.400000,NaN,NaN,NaN,NaN,38.000000,NaN,NaN


In [7]:
# Checking if missing data or null values are present in the dataset

df.isnull().sum()

customer_id                0
age                        0
gender                     0
item_purchased             0
category                   0
purchase_amount            0
location                   0
size                       0
color                      0
season                     0
review_rating             37
subscription_status        0
shipping_type              0
discount_applied           0
promo_code_used            0
previous_purchases         0
payment_method             0
frequency_of_purchases     0
dtype: int64

In [9]:
# Imputing missing values in Review Rating column with the median rating of the product category

df['review_rating'] = df.groupby('category')['review_rating'].transform(
    lambda x: x.fillna(x.median())
)

In [11]:
df.isnull().sum()

customer_id               0
age                       0
gender                    0
item_purchased            0
category                  0
purchase_amount           0
location                  0
size                      0
color                     0
season                    0
review_rating             0
subscription_status       0
shipping_type             0
discount_applied          0
promo_code_used           0
previous_purchases        0
payment_method            0
frequency_of_purchases    0
dtype: int64

In [12]:
# create a new column age_group
labels = ['Young Adult', 'Adult', 'Middle-aged', 'Senior']
df['age_group'] = pd.qcut(df['age'], q=4, labels = labels)

In [13]:
df[['age','age_group']].head(10)

,age,age_group
0,55,Middle-aged
1,19,Young Adult
2,50,Middle-aged
3,21,Young Adult
4,45,Middle-aged
5,46,Middle-aged
6,63,Senior
7,27,Young Adult
8,26,Young Adult
9,57,Middle-aged


In [14]:
# create new column purchase_frequency_days

frequency_mapping = {
    'Fortnightly': 14,
    'Weekly': 7,
    'Monthly': 30,
    'Quarterly': 90,
    'Bi-Weekly': 14,
    'Annually': 365,
    'Every 3 Months': 90
}

df['purchase_frequency_days'] = df['frequency_of_purchases'].map(frequency_mapping)

In [15]:
df[['purchase_frequency_days','frequency_of_purchases']].head(10)

,purchase_frequency_days,frequency_of_purchases
0,14,Fortnightly
1,14,Fortnightly
2,7,Weekly
3,7,Weekly
4,365,Annually
5,7,Weekly
6,90,Quarterly
7,7,Weekly
8,365,Annually
9,90,Quarterly


In [16]:
df[['discount_applied','promo_code_used']].head(10)

,discount_applied,promo_code_used
0,Yes,Yes
1,Yes,Yes
2,Yes,Yes
3,Yes,Yes
4,Yes,Yes
5,Yes,Yes
6,Yes,Yes
7,Yes,Yes
8,Yes,Yes
9,Yes,Yes


In [17]:
(df['discount_applied'] == df['promo_code_used']).all()

np.True_

In [18]:
# Dropping promo code used column

df = df.drop('promo_code_used', axis=1)

In [19]:
df.columns

Index(['customer_id', 'age', 'gender', 'item_purchased', 'category',
       'purchase_amount', 'location', 'size', 'color', 'season',
       'review_rating', 'subscription_status', 'shipping_type',
       'discount_applied', 'previous_purchases', 'payment_method',
       'frequency_of_purchases', 'age_group', 'purchase_frequency_days'],
      dtype='str')

In [20]:
df.to_csv("cleaned_customer_data.csv", index=False)

In [22]:
# Export to MySQL database using db.py connection utility
engine = get_db_engine()
if engine is not None:
    try:
        df.to_sql("customer", engine, if_exists="replace", index=False)
        print(" Successfully exported cleaned dataset to MySQL table 'customer'.")
        
        # Verify read-back
        sample_query = pd.read_sql("SELECT * FROM customer LIMIT 3;", engine)
        display(sample_query)
    except Exception as e:
        print(f"⚠️ MySQL Export failed: {e}")

 Successfully exported cleaned dataset to MySQL table 'customer'.


,customer_id,age,gender,item_purchased,category,purchase_amount,location,size,color,season,review_rating,subscription_status,shipping_type,discount_applied,previous_purchases,payment_method,frequency_of_purchases,age_group,purchase_frequency_days
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Express,Yes,14,Venmo,Fortnightly,Middle-aged,14
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Express,Yes,2,Cash,Fortnightly,Young Adult,14
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,Yes,Free Shipping,Yes,23,Credit Card,Weekly,Middle-aged,7


In [23]:



print("--- 1. Target Variable Distribution ---")
print(df['subscription_status'].value_counts(dropna=False))
print("\nPercentage Split:")
print(df['subscription_status'].value_counts(normalize=True) * 100)

print("\n--- 2. Checking Shipping Type vs Subscription Status ---")
print(pd.crosstab(df['subscription_status'], df['shipping_type'], normalize='index') * 100)

print("\n--- 3. Checking Discount Applied vs Subscription Status ---")
print(pd.crosstab(df['subscription_status'], df['discount_applied'], normalize='index') * 100)

print("\n--- 4. Summary Statistics for Numerical Features ---")
print(df[['age', 'purchase_amount', 'review_rating', 'previous_purchases', 'purchase_frequency_days']].describe())

--- 1. Target Variable Distribution ---
subscription_status
No     2847
Yes    1053
Name: count, dtype: int64

Percentage Split:
subscription_status
No     73.0
Yes    27.0
Name: proportion, dtype: float64

--- 2. Checking Shipping Type vs Subscription Status ---
shipping_type        2-Day Shipping    Express  Free Shipping  Next Day Air  \
subscription_status                                                           
No                        16.649104  16.016860      17.492097     16.859852   
Yes                       14.529915  18.043685      16.809117     15.954416   

shipping_type         Standard  Store Pickup  
subscription_status                           
No                   16.824728     16.157359  
Yes                  16.619183     18.043685  

--- 3. Checking Discount Applied vs Subscription Status ---
discount_applied            No         Yes
subscription_status                       
No                   78.082192   21.917808
Yes                   0.000000  100.00000